In [5]:
import pandas as pd
import difflib
import re

def clean_text(text):
    """Clean and normalize text for matching"""
    if not isinstance(text, str):
        return ""
    # Convert to lowercase
    text = str(text).lower().strip()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove punctuation that might cause matching issues
    text = re.sub(r'[.,;:!?"\'-]', ' ', text)
    # Remove multiple spaces again after punctuation removal
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def calculate_similarity(str1, str2):
    """Calculate similarity ratio between two strings"""
    return difflib.SequenceMatcher(None, str1, str2).ratio()

def get_first_n_words(text, n=4):
    """Extract first n words from text"""
    if not isinstance(text, str):
        return ""
    words = re.findall(r'\b\w+\b', text.lower())
    # Return all words if less than n are available
    return ' '.join(words[:min(n, len(words))])

def combine_award_datasets(recipients_csv, stats_csv, output_file='combined_awards.csv', 
                          similarity_threshold=0.85, word_count=4, exact_match_first=True):

    print("=" * 60)
    print("STARTING AWARD MATCHING PROCESS")
    print("=" * 60)
    
    # Load the datasets
    print(f"Attempting to load recipients data from: {recipients_csv}")
    try:
        recipients_df = pd.read_csv(recipients_csv)
        print(f"✓ Successfully loaded recipients data: {len(recipients_df)} rows")
    except Exception as e:
        print(f"❌ ERROR loading recipients data: {str(e)}")
        raise
    
    print(f"Attempting to load award statistics from: {stats_csv}")
    try:
        stats_df = pd.read_csv(stats_csv)
        print(f"✓ Successfully loaded statistics data: {len(stats_df)} rows")
    except Exception as e:
        print(f"❌ ERROR loading statistics data: {str(e)}")
        raise
    
    print("\nCleaning column names in both datasets...")
    
    # Clean column names
    recipients_df.columns = recipients_df.columns.str.strip()
    stats_df.columns = stats_df.columns.str.strip()
    
    # Identify award name columns
    print("Looking for award name column in recipients dataset...")
    for col in ['clean_award_name', 'awardname']:
        print(f"  - Checking for column '{col}'...")
        if col in recipients_df.columns:
            recipient_award_col = col
            print(f"  ✓ Found award name column: '{recipient_award_col}'")
            break
    else:
        recipient_award_col = None
        
    if not recipient_award_col:
        print("❌ ERROR: Could not find award name column in recipients dataset")
        print(f"Available columns: {list(recipients_df.columns)}")
        raise ValueError("Could not find award name column in recipients dataset")
    
    # Check for the focal and pathway award columns
    print("\nChecking for required columns in statistics dataset...")
    missing_cols = []
    for required_col in ['Focal Award', 'Pathway Award']:
        print(f"  - Checking for column '{required_col}'...")
        if required_col not in stats_df.columns:
            missing_cols.append(required_col)
            print(f"  ❌ Column '{required_col}' not found")
        else:
            print(f"  ✓ Column '{required_col}' found")
            
    if missing_cols:
        print(f"❌ ERROR: Missing required columns in stats dataset: {missing_cols}")
        print(f"Available columns: {list(stats_df.columns)}")
        raise ValueError(f"Could not find '{missing_cols}' columns in stats dataset")
    
    print(f"\nUsing '{recipient_award_col}' from recipients dataset for matching")
    
    # Process the stats dataset to split pathway awards
    print("\nProcessing statistics dataset to split awards with pipes (|)...")
    stats_processed = []
    pipe_count = 0
    
    for idx, row in stats_df.iterrows():
        if idx == 0 or (idx + 1) % 50 == 0 or idx == len(stats_df) - 1:
            print(f"  Processing row {idx + 1}/{len(stats_df)} ({(idx + 1)/len(stats_df)*100:.1f}% complete)")
            
        new_row = row.copy()
        
        # Split the focal award if it contains a pipe
        focal_award = row['Focal Award']
        if isinstance(focal_award, str) and '|' in focal_award:
            parts = [p.strip() for p in focal_award.split('|')]
            new_row['focal_part_1'] = parts[0]
            if len(parts) > 1:
                new_row['focal_part_2'] = parts[1]
            pipe_count += 1
        else:
            new_row['focal_part_1'] = focal_award
            new_row['focal_part_2'] = None
            
        # Split the pathway award if it contains a pipe
        pathway_award = row['Pathway Award']
        if isinstance(pathway_award, str) and '|' in pathway_award:
            parts = [p.strip() for p in pathway_award.split('|')]
            new_row['pathway_part_1'] = parts[0]
            if len(parts) > 1:
                new_row['pathway_part_2'] = parts[1]
            pipe_count += 1
        else:
            new_row['pathway_part_1'] = pathway_award
            new_row['pathway_part_2'] = None
            
        stats_processed.append(new_row)
    
    print(f"  ✓ Processed {len(stats_processed)} rows, found {pipe_count} awards with pipes")
    
    # Convert to DataFrame
    stats_processed_df = pd.DataFrame(stats_processed)
    print(f"  ✓ Created processed statistics DataFrame with {len(stats_processed_df)} rows")
    
    # Pre-compute the first N words for partial matching
    if word_count > 0:
        print(f"\nPre-computing first {word_count} words for all awards...")
        word_count_start_time = pd.Timestamp.now()
        
        # Pre-compute first N words for each award
        print("  Computing first words for recipient awards...")
        for idx, recipient in recipients_df.iterrows():
            if idx == 0 or (idx + 1) % 100 == 0 or idx == len(recipients_df) - 1:
                print(f"    Processing {idx + 1}/{len(recipients_df)} ({(idx + 1)/len(recipients_df)*100:.1f}% complete)")
                
            award_name = recipient[recipient_award_col]
            recipient['first_n_words'] = get_first_n_words(award_name, word_count)
        
        # Pre-compute first N words for each stats award
        print("  Computing first words for statistics awards...")
        part_cols = ['focal_part_1', 'focal_part_2', 'pathway_part_1', 'pathway_part_2', 
                   'Focal Award', 'Pathway Award']
        total_parts = len(stats_processed_df) * len(part_cols)
        processed_parts = 0
        
        for idx, stats in stats_processed_df.iterrows():
            for part_col in part_cols:
                processed_parts += 1
                if processed_parts == 1 or processed_parts % 200 == 0 or processed_parts == total_parts:
                    print(f"    Processing {processed_parts}/{total_parts} ({processed_parts/total_parts*100:.1f}% complete)")
                
                if part_col in stats and pd.notna(stats[part_col]):
                    stats[f'{part_col}_first_{word_count}'] = get_first_n_words(stats[part_col], word_count)
                    
        word_count_end_time = pd.Timestamp.now()
        duration = (word_count_end_time - word_count_start_time).total_seconds()
        print(f"  ✓ Completed first word computation in {duration:.2f} seconds")
    
    # Now process each recipient record to find matches
    results = []
    total_records = len(recipients_df)
    
    print(f"\nProcessing {total_records} recipient records...")
    
    for index, recipient in recipients_df.iterrows():
        # Print progress indicator
        if (index + 1) % 10 == 0 or index == 0 or index == total_records - 1:
            print(f"Processing record {index + 1}/{total_records} ({(index + 1)/total_records*100:.1f}% complete)")
            
        award_name = recipient[recipient_award_col]
        award_name_clean = clean_text(award_name)
        award_first_words = recipient['first_n_words'] if 'first_n_words' in recipient else ""
        
        print(f"\nLooking for matches for: '{award_name}'")
        
        match_found = False
        matched_stats = None
        matched_part = None
        match_type = None
        match_method = None
        best_similarity = 0.0
        
        # Try exact match first if requested
        if exact_match_first:
            for _, stats in stats_processed_df.iterrows():
                # Check against all possible parts
                parts_to_check = [
                    ('focal_part_1', 'Focal Award (Part 1)'),
                    ('focal_part_2', 'Focal Award (Part 2)'),
                    ('pathway_part_1', 'Pathway Award (Part 1)'),
                    ('pathway_part_2', 'Pathway Award (Part 2)'),
                    ('Focal Award', 'Focal Award (Full)'),
                    ('Pathway Award', 'Pathway Award (Full)')
                ]
                
                for part_col, part_type in parts_to_check:
                    if part_col in stats and pd.notna(stats[part_col]):
                        part_clean = clean_text(stats[part_col])
                        
                        # Check for exact match after cleaning
                        if award_name_clean == part_clean:
                            match_found = True
                            matched_stats = stats
                            matched_part = stats[part_col]
                            match_type = part_type
                            match_method = "Exact"
                            best_similarity = 1.0
                            print(f"  ✓ EXACT MATCH found with '{stats[part_col]}' as {part_type}")
                            break
                
                if match_found:
                    break
        
        # Try partial match by first N words if no exact match yet
        if not match_found and word_count > 0 and award_first_words:
            for _, stats in stats_processed_df.iterrows():
                parts_to_check = [
                    ('focal_part_1', 'Focal Award (Part 1)'),
                    ('focal_part_2', 'Focal Award (Part 2)'),
                    ('pathway_part_1', 'Pathway Award (Part 1)'),
                    ('pathway_part_2', 'Pathway Award (Part 2)'),
                    ('Focal Award', 'Focal Award (Full)'),
                    ('Pathway Award', 'Pathway Award (Full)')
                ]
                
                for part_col, part_type in parts_to_check:
                    first_words_col = f'{part_col}_first_{word_count}'
                    if part_col in stats and pd.notna(stats[part_col]) and first_words_col in stats:
                        part_first_words = stats[first_words_col]
                        
                        # Check for first N words match
                        if part_first_words and award_first_words == part_first_words:
                            match_found = True
                            matched_stats = stats
                            matched_part = stats[part_col]
                            match_type = part_type
                            match_method = f"First {word_count} Words"
                            best_similarity = 0.95  # Arbitrary high value for first words match
                            print(f"  ✓ PARTIAL MATCH (first words) found with '{stats[part_col]}' as {part_type}")
                            print(f"    First {word_count} words: '{award_first_words}'")
                            break
                
                if match_found:
                    break
                
        # If still no match, try fuzzy matching
        if not match_found:
            for _, stats in stats_processed_df.iterrows():
                parts_to_check = [
                    ('focal_part_1', 'Focal Award (Part 1)'),
                    ('focal_part_2', 'Focal Award (Part 2)'),
                    ('pathway_part_1', 'Pathway Award (Part 1)'),
                    ('pathway_part_2', 'Pathway Award (Part 2)'),
                    ('Focal Award', 'Focal Award (Full)'),
                    ('Pathway Award', 'Pathway Award (Full)')
                ]
                
                for part_col, part_type in parts_to_check:
                    if part_col in stats and pd.notna(stats[part_col]):
                        part_clean = clean_text(stats[part_col])
                        
                        # Calculate similarity between the strings
                        similarity = calculate_similarity(award_name_clean, part_clean)
                        
                        # If similarity is above threshold and better than previous matches
                        if similarity >= similarity_threshold and similarity > best_similarity:
                            match_found = True
                            matched_stats = stats
                            matched_part = stats[part_col]
                            match_type = part_type
                            match_method = "Fuzzy"
                            best_similarity = similarity
                            print(f"  ✓ FUZZY MATCH found with '{stats[part_col]}' as {part_type}")
                            print(f"    Similarity score: {similarity:.4f}")
            
        # Create result record
        result = recipient.copy()
        
        # Add match information
        if match_found:
            result['match_found'] = True
            result['matched_with'] = matched_part
            result['match_type'] = match_type
            result['match_method'] = match_method
            result['similarity_score'] = best_similarity
            
            # Add all stats fields
            if matched_stats is not None:
                for col, val in matched_stats.items():
                    if col not in ['focal_part_1', 'focal_part_2', 'pathway_part_1', 'pathway_part_2']:
                        result[f'stats_{col}'] = val
            
            match_desc = f"{match_method} match found: '{award_name}' matches '{matched_part}' as {match_type}"
            print(f"✅ MATCH CONFIRMED: {match_desc}")
            
            # Print some additional info for debugging
            if 'Discipline' in matched_stats:
                print(f"  Discipline: {matched_stats['Discipline']}")
            if 'Focal Award' in matched_stats:
                print(f"  Focal Award: {matched_stats['Focal Award']}")
        else:
            result['match_found'] = False
            result['matched_with'] = None
            result['match_type'] = 'No Match'
            result['match_method'] = None
            result['similarity_score'] = 0.0
            
            print(f"❌ NO MATCH FOUND for '{award_name}'")
        
        results.append(result)
    
    # Convert to DataFrame and save
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_file, index=False)
    
    # Print match statistics
    matches_found = results_df['match_found'].sum()
    total_records = len(results_df)
    match_percentage = (matches_found / total_records) * 100 if total_records > 0 else 0
    
    print(f"\n===== MATCHING RESULTS SUMMARY =====")
    print(f"Total records processed: {total_records}")
    print(f"Matches found: {matches_found} ({match_percentage:.1f}%)")
    print(f"Unmatched records: {total_records - matches_found} ({100 - match_percentage:.1f}%)")
    
    # Show matches by type and method
    if matches_found > 0:
        print("\n----- Matches by Type -----")
        match_types = results_df[results_df['match_found']]['match_type'].value_counts()
        for match_type, count in match_types.items():
            percentage = (count / matches_found) * 100
            print(f"  • {match_type}: {count} ({percentage:.1f}%)")
        
        print("\n----- Matches by Method -----")
        match_methods = results_df[results_df['match_found']]['match_method'].value_counts()
        for match_method, count in match_methods.items():
            percentage = (count / matches_found) * 100
            print(f"  • {match_method}: {count} ({percentage:.1f}%)")
        
        # Output list of unmatched awards
        if total_records - matches_found > 0:
            unmatched = results_df[~results_df['match_found']]
            print(f"\n----- Unmatched Awards ({len(unmatched)}) -----")
            for i, (_, row) in enumerate(unmatched.iterrows()):
                print(f"  {i+1}. '{row[recipient_award_col]}'")
                if i >= 19 and len(unmatched) > 20:  # Show only first 20
                    print(f"  ... and {len(unmatched) - 20} more")
                    break
    
    return results_df

if __name__ == "__main__":
    print("\n" + "=" * 80)
    print("AWARD MATCHING SCRIPT - START")
    print("=" * 80)
    
    import os
    import time
    
    # Check what files are available in the current directory
    print("\nListing files in current directory:")
    files = os.listdir(".")
    for file in files:
        print(f"  - {file}")
    
    print("\nListing files in ../clean_dataset directory (if it exists):")
    try:
        clean_dataset_files = os.listdir("../clean_dataset")
        for file in clean_dataset_files:
            print(f"  - {file}")
    except FileNotFoundError:
        print("  ❌ ../clean_dataset directory not found")
    except Exception as e:
        print(f"  ❌ Error accessing ../clean_dataset: {str(e)}")
    
    recipients_file = "../clean_dataset/cleaned_ri_matches_cleaning.csv"
    stats_file = "../clean_dataset/cleaned_discipline_specific_pathways.csv"
    
    print(f"\nWill attempt to load recipients from: {recipients_file}")
    print(f"Will attempt to load statistics from: {stats_file}")
    
    start_time = time.time()
    
    try:
        combined_df = combine_award_datasets(
            recipients_file, 
            stats_file,
            similarity_threshold=0.85,  # Match strings that are at least 85% similar
            word_count=4,              # Use first 4 words for word matching
            exact_match_first=True     # Try exact matching first
        )
        
        print("\nSample of combined dataset:")
        print(combined_df.head())
        
        end_time = time.time()
        duration = end_time - start_time
        print(f"\nTotal processing time: {duration:.2f} seconds ({duration/60:.2f} minutes)")
        print("\n" + "=" * 80)
        print("AWARD MATCHING SCRIPT - COMPLETE")
        print("=" * 80)
        
    except Exception as e:
        print(f"\n❌ ERROR: Script execution failed: {str(e)}")
        import traceback
        print("\nDetailed error traceback:")
        traceback.print_exc()
        print("\n" + "=" * 80)
        print("AWARD MATCHING SCRIPT - FAILED")
        print("=" * 80)
    
    print("\nSample of combined dataset:")
    print(combined_df.head())


AWARD MATCHING SCRIPT - START

Listing files in current directory:
  - .DS_Store
  - combine_dataset.ipynb

Listing files in ../clean_dataset directory (if it exists):
  - cleaned_ri_matches_cleaning.csv
  - cleaned_discipline_specific_pathways.csv

Will attempt to load recipients from: ../clean_dataset/cleaned_ri_matches_cleaning.csv
Will attempt to load statistics from: ../clean_dataset/cleaned_discipline_specific_pathways.csv
STARTING AWARD MATCHING PROCESS
Attempting to load recipients data from: ../clean_dataset/cleaned_ri_matches_cleaning.csv
✓ Successfully loaded recipients data: 2403 rows
Attempting to load award statistics from: ../clean_dataset/cleaned_discipline_specific_pathways.csv
✓ Successfully loaded statistics data: 304599 rows

Cleaning column names in both datasets...
Looking for award name column in recipients dataset...
  - Checking for column 'clean_award_name'...
  ✓ Found award name column: 'clean_award_name'

Checking for required columns in statistics dataset

In [8]:
import pandas as pd

# Load the CSV file
file_path = "combined_awards.csv"
df = pd.read_csv(file_path)

# Drop unwanted columns
columns_to_remove = [
    "awardgoverningsocietyid", "awardid", "awardreceivedid", "AAUID",
    "clientfacultyid", "orcid", "institutionname", "asofdate", "asofdate_iso",
    "is_historical", "possible_duplicate", "match_found", "matched_with",
    "match_type", "match_method", "similarity_score", "clean_award_name"
]
df = df.drop(columns=columns_to_remove, errors="ignore")

# Drop rows where 'awardreceivedawardyear' or 'award_age' are NaN
df = df.dropna(subset=["awardreceivedawardyear", "award_age"])

# Convert award year and award age to integers
df["awardreceivedawardyear"] = df["awardreceivedawardyear"].astype(int)
df["award_age"] = df["award_age"].astype(int)

# Save the cleaned file
cleaned_file_path = "cleaned_combined_awards.csv"
df.to_csv(cleaned_file_path, index=False, sep='\t')

print(f"Cleaned CSV saved as {cleaned_file_path}")


Cleaned CSV saved as cleaned_combined_awards.csv
